> ⚠️ **WARNING — UNDER CONSTRUCTION**
>
> This notebook is a work in progress. **Use at your own risk.**
> It may contain errors, incomplete sections, or incorrect results.
> No guarantees are made about its accuracy or completeness.

# The Angular Power Spectrum of Galaxies
## A practical introduction using the DESI BGS Legacy Survey DR10 sample

**Data:** Comparat et al. (2025), A&A 697, A173  
**arXiv:** [2503.19796](https://arxiv.org/abs/2503.19796)  
**DOI:** [10.1051/0004-6361/202554208](https://doi.org/10.1051/0004-6361/202554208)

---

## Learning objectives

After this tutorial you will be able to:
1. Understand the concept of the **angular power spectrum** $C_\ell$ and its relation to galaxy clustering.
2. Pixelise a galaxy catalog onto a **HEALPix** sphere map and build the overdensity field.
3. Measure $C_\ell$ using the **pseudo-$C_\ell$** (anafast) method with shot-noise subtraction.
4. Estimate the **Gaussian covariance** of $C_\ell$.
5. Compare with a theoretical **Limber approximation** prediction from the HOD model (`hod_mod`).

---

## 1. Theoretical background

### 1.1 Galaxy clustering and the density contrast

Galaxies are not uniformly distributed — they cluster along dark matter filaments and avoid voids. We quantify the **galaxy overdensity** on the sphere as:

$$\delta(\hat{n}) = \frac{n(\hat{n}) - \bar{n}}{\bar{n}}$$

where $n(\hat{n})$ is the galaxy number count in a sky pixel at direction $\hat{n}$ and $\bar{n}$ is the mean galaxy count per pixel.

### 1.2 The angular power spectrum $C_\ell$

We decompose the overdensity field into **spherical harmonics**:

$$\delta(\hat{n}) = \sum_{\ell=0}^{\infty}\sum_{m=-\ell}^{\ell} a_{\ell m}\,Y_{\ell m}(\hat{n})$$

The **angular power spectrum** is defined as the variance of the spherical harmonic coefficients:

$$C_\ell = \frac{1}{2\ell+1}\sum_{m=-\ell}^{\ell}|a_{\ell m}|^2$$

Each multipole $\ell$ corresponds to an angular scale $\theta \approx 180°/\ell$. Large $\ell$ means small angular scales.

| Multipole $\ell$ | Angular scale $\theta$ |
|------------------|-----------------------|
| 10 | ~18° |
| 100 | ~1.8° |
| 1000 | ~0.18° |

### 1.3 Relation to the 3D power spectrum (Limber approximation)

For a galaxy sample with redshift distribution $n(z)$ (normalised to $\bar{n} = \int n(z)dz$), the angular power spectrum is related to the 3D galaxy power spectrum $P_{gg}(k)$ via the **Limber approximation** (Limber 1953, ApJ 117, 134):

$$C_\ell = \int_0^\infty \frac{H(z)}{c}\,\left[\frac{n(z)}{\bar{n}}\right]^2 \frac{P_{gg}\!\left(\frac{\ell+1/2}{\chi(z)},\,z\right)}{\chi^2(z)}\,dz$$

where:
- $H(z)$ is the Hubble parameter
- $\chi(z)$ is the comoving distance
- $n(z)/\bar{n}$ is the normalised redshift selection function
- $P_{gg}(k,z) = b^2_{\rm eff}(z)\,P_{\rm lin}(k,z)$ in the **2-halo approximation** (large scales)
- $b_{\rm eff}$ is the effective **galaxy bias** (see More et al. 2015, ApJ 806, 2)

The Limber approximation is accurate for $\ell \gtrsim 10$ and narrow $n(z)$.

### 1.4 Shot noise

A discrete galaxy sample has irreducible **shot noise** (Poisson fluctuations):

$$N_\ell = \frac{\Omega_{\rm pix}}{N_{\rm gal}}$$

where $\Omega_{\rm pix}$ is the pixel area in steradians and $N_{\rm gal}$ is the total (weighted) galaxy count. This adds a white-noise floor to $C_\ell$ that must be subtracted.

### 1.5 Gaussian covariance

In the Gaussian approximation, the variance on each band-power estimate is:

$$\mathrm{Var}(\hat{C}_\ell) = \frac{2\,(C_\ell + N_\ell)^2}{(2\ell+1)\,f_{\rm sky}\,N_{\rm modes}}$$

where $N_{\rm modes}$ is the number of modes in the $\ell$-bin and $f_{\rm sky} = \Omega_{\rm survey}/4\pi$ is the sky fraction covered.

### 1.6 The pseudo-$C_\ell$ method

Because galaxy surveys cover only a **fraction of the sky**, we measure the **pseudo-$C_\ell$** (or `anafast` spectrum of the masked map) and correct for the mode-coupling introduced by the mask. The correction factor at lowest order is $f_{\rm sky}$:

$$\hat{C}_\ell^{\rm corrected} \approx \frac{\tilde{C}_\ell}{f_{\rm sky}}$$

More accurate corrections (the MASTER algorithm) account for off-diagonal mode coupling. For a pedagogical introduction, see Hivon et al. (2002), ApJ 567, 2.

---

## 2. Setup and imports

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.cosmology import FlatLambdaCDM
import healpy as hp

# sum_stat
from sum_stat.catalogue import GalaxyCatalogue
from sum_stat.powspec.angular_cl import cl_angular

# hod_mod — HOD clustering prediction
from hod_mod.cosmology.power_spectrum import LinearPowerSpectrum
from hod_mod.cosmology.halo_mass_function import HaloMassFunction
from hod_mod.galaxies.hod import HODModel
from hod_mod.galaxies.clustering import HODClusteringPrediction

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 13,
    'axes.labelsize': 14,
    'legend.fontsize': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

print('Imports OK')

---

## 3. Loading the data

We use the same BGS LS10 volume-limited sample as in the SMF notebook:
- $\log_{10}(M_*/M_\odot) > 10.0$, $0.05 < z < 0.18$
- 2,759,238 galaxies in the data catalog
- 13,795,884 random points in the random catalog

The **random catalog** is essential for measuring the angular power spectrum: it provides a uniform representation of the survey window function (footprint + mask) against which the galaxy distribution is compared.

**FITS columns:**
| File | Columns |
|------|---------|
| DATA | RA, DEC, EBV, LPH_MASS_BEST, BEST_Z |
| RAND | RA, DEC, EBV, Z |

In [ ]:
DATA_FILE = ('/home/comparat/data/legacysurvey/dr10/sweep/BGS_VLIM_Mstar/'
             'LS10_VLIM_ANY_10.0_Mstar_12.0_0.05_z_0.18_N_2759238_DATA.fits')
RAND_FILE = ('/home/comparat/data/legacysurvey/dr10/sweep/BGS_VLIM_Mstar/'
             'LS10_VLIM_ANY_10.0_Mstar_12.0_0.05_z_0.18_N_2759238_RAND.fits')

# --- Data ---
with fits.open(DATA_FILE) as hdul:
    d = hdul[1].data
    ra    = d['RA'].astype(np.float64)
    dec   = d['DEC'].astype(np.float64)
    z     = d['BEST_Z'].astype(np.float64)
    mstar = d['LPH_MASS_BEST'].astype(np.float64)
    print(f'DATA: {len(ra):,} galaxies')

# --- Randoms ---
with fits.open(RAND_FILE) as hdul:
    r = hdul[1].data
    rand_ra  = r['RA'].astype(np.float64)
    rand_dec = r['DEC'].astype(np.float64)
    rand_z   = r['Z'].astype(np.float64)
    print(f'RAND: {len(rand_ra):,} random points')

# Apply redshift and stellar mass cuts
Z_MIN = 0.05
Z_MAX = 0.18
MSTAR_LIMIT = 10.0

mask_data = (z >= Z_MIN) & (z <= Z_MAX) & (mstar >= MSTAR_LIMIT)
ra_g    = ra[mask_data]
dec_g   = dec[mask_data]
z_g     = z[mask_data]

mask_rand = (rand_z >= Z_MIN) & (rand_z <= Z_MAX)
rand_ra_g  = rand_ra[mask_rand]
rand_dec_g = rand_dec[mask_rand]

print(f'\nAfter cuts:')
print(f'  Galaxies : {len(ra_g):,}')
print(f'  Randoms  : {len(rand_ra_g):,}')

### 3.1 Redshift distribution n(z)

The redshift distribution $n(z)$ enters the Limber approximation and must be measured from the data. We also check that the random catalog follows the same $n(z)$.

In [ ]:
z_bins = np.linspace(Z_MIN, Z_MAX, 30)
z_centres = 0.5 * (z_bins[:-1] + z_bins[1:])

nz_data, _ = np.histogram(z_g, bins=z_bins)
nz_rand, _ = np.histogram(rand_z[mask_rand], bins=z_bins)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(z_centres, nz_data, width=np.diff(z_bins), alpha=0.7, color='steelblue', label='Galaxies (data)')
ax.bar(z_centres, nz_rand / 5, width=np.diff(z_bins), alpha=0.5, color='tomato',
       label='Randoms / 5')
ax.set_xlabel('Photometric redshift $z$')
ax.set_ylabel('Count per bin')
ax.set_title('Redshift distribution $n(z)$')
ax.legend()
plt.tight_layout()
plt.show()

# Store the normalised n(z) for the Limber integral
nz_norm = nz_data / nz_data.sum()  # normalise to sum=1
dz = np.diff(z_bins)[0]
print(f'Mean redshift: z_mean = {np.sum(z_centres * nz_norm):.3f}')

---

## 4. Building the HEALPix galaxy map

### 4.1 What is HEALPix?

**HEALPix** (Hierarchical Equal Area isoLatitude PIXelization, Górski et al. 2005, ApJ 622, 759) divides the sphere into $N_{\rm pix} = 12\,N_{\rm side}^2$ pixels of **equal area** $\Omega_{\rm pix} = 4\pi/N_{\rm pix}$.

| $N_{\rm side}$ | $N_{\rm pix}$ | $\Omega_{\rm pix}$ [deg²] | $\theta_{\rm pix}$ |
|---------------|--------------|--------------------------|-------------------|
| 64 | 49,152 | 0.84 | ~55' |
| 128 | 196,608 | 0.21 | ~27' |
| 256 | 786,432 | 0.052 | ~14' |
| 512 | 3,145,728 | 0.013 | ~7' |
| 1024 | 12,582,912 | 0.0033 | ~3.5' |

### 4.2 Choosing $N_{\rm side}$

The choice of $N_{\rm side}$ sets the minimum angular scale we can probe: $\ell_{\rm max} \approx 2\,N_{\rm side}$ (Nyquist limit). We also need enough galaxies per pixel to avoid Poisson-dominated pixels.

With $N_{\rm gal} \approx 2.7\times10^6$ galaxies over $A \approx 16{,}800\,{\rm deg}^2$:
- Mean number density: $\bar{n} \approx 160\,{\rm gal/deg}^2$
- At $N_{\rm side}=512$ (pixel area $= 0.013\,{\rm deg}^2$): $\bar{n}_{\rm pix} \approx 2$ galaxies/pixel
- At $N_{\rm side}=256$ (pixel area $= 0.052\,{\rm deg}^2$): $\bar{n}_{\rm pix} \approx 8$ galaxies/pixel

We use $N_{\rm side}=512$ to access scales up to $\ell \sim 1000$, accepting that shot noise dominates at high $\ell$.

In [ ]:
NSIDE = 512
N_PIX = hp.nside2npix(NSIDE)
OMEGA_PIX = hp.nside2pixarea(NSIDE)  # sr
OMEGA_PIX_DEG2 = hp.nside2pixarea(NSIDE, degrees=True)

print(f'HEALPix parameters:')
print(f'  Nside     = {NSIDE}')
print(f'  N_pix     = {N_PIX:,}')
print(f'  Omega_pix = {OMEGA_PIX_DEG2:.4f} deg²')
print(f'  ell_max   ~ {2*NSIDE} (Nyquist)')

# Pixelise galaxy catalog
pix_gal  = hp.ang2pix(NSIDE, ra_g,  dec_g,  lonlat=True)
pix_rand = hp.ang2pix(NSIDE, rand_ra_g, rand_dec_g, lonlat=True)

# Galaxy count map
n_map = np.zeros(N_PIX)
for p in pix_gal:
    n_map[p] += 1

print(f'\nGalaxy map statistics:')
n_nonzero = (n_map > 0).sum()
print(f'  Occupied pixels: {n_nonzero:,} / {N_PIX:,}')
print(f'  Mean galaxies per occupied pixel: {n_map[n_map>0].mean():.1f}')
print(f'  Max galaxies in one pixel: {int(n_map.max())}')

### 4.3 Building the survey mask

The survey **mask** identifies which pixels are part of the observed footprint. We build it from the random catalog: a pixel is considered "observed" if it contains at least one random point.

We also apply a **Galactic plane cut** ($|b| > 20°$) to remove regions with heavy dust extinction, and a **bright star mask** (automatic, since the randoms already encode this).

In [ ]:
# Build mask from randoms
rand_map = np.zeros(N_PIX)
for p in pix_rand:
    rand_map[p] += 1

mask = (rand_map > 0).astype(np.float64)

# Survey statistics
f_sky = mask.sum() / N_PIX
area_deg2 = mask.sum() * OMEGA_PIX_DEG2
area_sr   = mask.sum() * OMEGA_PIX

print(f'Survey mask statistics:')
print(f'  Observed pixels : {int(mask.sum()):,}')
print(f'  f_sky           : {f_sky:.4f} ({f_sky*100:.2f}% of sky)')
print(f'  Area            : {area_deg2:,.0f} deg²')
print(f'  Area            : {area_sr:.2f} sr')

# Visualise the mask
hp.mollview(mask, title='Survey Mask (white=observed, black=masked)',
            cmap='gray_r', min=0, max=1)
plt.savefig('aps_healpix_mask.pdf', bbox_inches='tight')
plt.show()

### 4.4 Building the galaxy overdensity map

The overdensity field is:

$$\delta_p = \frac{n_p}{\bar{n}} - 1$$

where $\bar{n} = N_{\rm gal} / N_{\rm pix,obs}$ is the mean galaxy count per observed pixel. Unobserved pixels are set to zero.

In [ ]:
# Mean density
n_pix_obs = mask.sum()
n_gal_eff = n_map[mask > 0].sum()
n_bar = n_gal_eff / n_pix_obs

# Overdensity map
delta_map = np.zeros(N_PIX)
obs = mask > 0
delta_map[obs] = n_map[obs] / n_bar - 1.0
delta_map *= mask  # zero out unobserved pixels

print(f'Overdensity map:')
print(f'  Mean galaxy count per pixel : {n_bar:.2f}')
print(f'  delta range                 : [{delta_map[obs].min():.2f}, {delta_map[obs].max():.2f}]')
print(f'  Mean delta (should be ~0)   : {delta_map[obs].mean():.4f}')

# Visualise the overdensity
hp.mollview(delta_map, title=r'Galaxy overdensity $\delta(\hat{n})$',
            cmap='RdBu_r', min=-3, max=3)
plt.savefig('aps_overdensity_map.pdf', bbox_inches='tight')
plt.show()

---

## 5. Measuring the angular power spectrum

We use `sum_stat.powspec.angular_cl.cl_angular()` which:
1. Pixelises the galaxy catalog onto a HEALPix map
2. Computes the pseudo-$C_\ell$ via `healpy.anafast`
3. Subtracts the shot noise $N_\ell = \Omega_{\rm pix}/N_{\rm gal}$
4. Applies the MASTER $f_{\rm sky}$ correction
5. Returns $\ell_{\rm eff}$, $C_\ell$, $N_\ell$, and the Gaussian covariance matrix

### 5.1 Choosing multipole bins

We bin $C_\ell$ in logarithmically spaced bins from $\ell_{\rm min}=10$ to $\ell_{\rm max}=2N_{\rm side}=1024$. The minimum $\ell$ is set by the survey footprint size:

$$\ell_{\rm min} \approx \pi / \theta_{\rm max} \approx \pi / \sqrt{\Omega_{\rm survey}} \approx \frac{180}{90} \approx 2$$

For safety we start at $\ell=10$ where the Limber approximation is also valid.

In [ ]:
# Build the GalaxyCatalogue
cat = GalaxyCatalogue(
    ra       = ra_g,
    dec      = dec_g,
    redshift = z_g,
    weight   = np.ones(len(ra_g)),
)

# Multipole bin edges — logarithmically spaced
ell_edges = np.unique(np.round(np.logspace(np.log10(10), np.log10(1100), 22)).astype(int))
print(f'Multipole bin edges: {ell_edges}')

print('\nComputing angular power spectrum C_ell...')
ell_eff, cl_data, nl_data, cov_data = cl_angular(
    gal                  = cat,
    nside                = NSIDE,
    ell_bins             = ell_edges,
    mask                 = mask,
    use_master_correction= True,   # apply f_sky correction
)
print('Done!')

cl_err = np.sqrt(np.diag(cov_data))

print(f'\nC_ell results:')
print(f'{"ell_eff":>8s}  {"C_ell":>14s}  {"N_ell":>14s}  {"sigma":>12s}  {"S/N":>6s}')
for le, cl, nl, se in zip(ell_eff, cl_data, nl_data, cl_err):
    snr = cl/se if se > 0 else 0
    flag = '***' if cl < 0 else ''
    print(f'{le:8.1f}  {cl:14.3e}  {nl:14.3e}  {se:12.3e}  {snr:6.1f} {flag}')

### 5.2 First look at the power spectrum

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Plot C_ell * ell*(ell+1) / (2*pi) — standard convention for CMB-like plots
norm = ell_eff * (ell_eff + 1) / (2 * np.pi)

good = cl_data > 0
ax.errorbar(ell_eff[good], (norm * cl_data)[good], yerr=(norm * cl_err)[good],
            fmt='o', color='steelblue', ms=6, capsize=4,
            label='Measured $C_\\ell$ (shot-noise subtracted)')

# Show shot noise
ax.plot(ell_eff, norm * nl_data, 'k:', lw=1.5, label='Shot noise $N_\\ell$')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(r'Multipole $\ell$', fontsize=14)
ax.set_ylabel(r'$\ell(\ell+1)C_\ell / (2\pi)$', fontsize=14)
ax.set_title(r'Angular Power Spectrum — BGS LS10 ($\log_{10}M_* > 10.0$, $0.05 < z < 0.18$)')
ax.legend()
ax.set_xlim(8, 1200)

plt.savefig('aps_cl_raw.pdf', bbox_inches='tight')
plt.show()

---

## 6. Theoretical prediction with `hod_mod`

### 6.1 The Halo Occupation Distribution (HOD)

In the **HOD framework** (Zheng et al. 2007, ApJ 667, 760), the number of galaxies in a halo of mass $M$ is parametrised as:

**Central galaxy occupation:**
$$\langle N_c \rangle(M) = \frac{1}{2}\left[1 + {\rm erf}\!\left(\frac{\log_{10}M - \log_{10}M_{\rm min}}{\sigma_{\log M}}\right)\right]$$

**Satellite galaxy occupation:**
$$\langle N_s \rangle(M) = \mathcal{H}(M - M_0)\,\left(\frac{M - M_0}{M_1}\right)^\alpha$$

where $\mathcal{H}$ is the Heaviside step function.

The **effective galaxy bias** at a given redshift is the halo-mass-weighted mean bias:

$$b_{\rm eff}(z) = \frac{\int (\langle N_c \rangle + \langle N_s \rangle)\,b(M,z)\,\frac{dn}{dM}\,dM}{\int (\langle N_c \rangle + \langle N_s \rangle)\,\frac{dn}{dM}\,dM}$$

where $b(M,z)$ is the halo bias function (Tinker et al. 2010, ApJ 724, 878) and $dn/dM$ is the halo mass function (Tinker et al. 2008, ApJ 688, 709).

### 6.2 HOD parameters for the BGS sample

For galaxies with $\log_{10}(M_*/M_\odot) > 10.0$, reasonable HOD parameters are (Zheng+2007 SDSS Main Sample calibration):

| Parameter | Value | Physical meaning |
|-----------|-------|------------------|
| $\log_{10}M_{\rm min}$ | 11.35 | Min. halo mass to host a central galaxy |
| $\sigma_{\log M}$ | 0.25 | Width of the central occupation step |
| $\log_{10}M_0$ | 11.20 | Satellite suppression mass scale |
| $\log_{10}M_1$ | 12.40 | Satellite mass scale |
| $\alpha$ | 1.0 | Satellite power-law slope |

These correspond to an effective bias $b_{\rm eff} \approx 1.2$ at $z \sim 0.1$, consistent with the large-scale clustering of BGS galaxies.

### 6.3 Limber approximation implementation

We implement the Limber approximation numerically:

$$C_\ell = \int_{z_{\rm min}}^{z_{\rm max}} \frac{H(z)}{c}\,\left[\frac{n(z)}{\bar{n}}\right]^2 b^2_{\rm eff}\,P_{\rm lin}\!\left(\frac{\ell+1/2}{\chi(z)}\right)\,\frac{dz}{\chi^2(z)}$$

Steps:
1. Set up the linear matter power spectrum $P_{\rm lin}(k)$ using `hod_mod.cosmology.power_spectrum.LinearPowerSpectrum` (CAMB)
2. Get the effective bias $b_{\rm eff}$ from `HODModel._integrate`
3. Compute the geometric factors $H(z)/c$ and $\chi(z)$ from the astropy cosmology
4. Integrate numerically over $z$

In [ ]:
# Cosmology (Planck 2018, Comparat et al. 2025)
cosmo_astropy = FlatLambdaCDM(H0=67.66, Om0=0.26069)

# hod_mod cosmology dictionary (same cosmology, h-unit convention)
theta_cosmo = {
    'h':              0.6766,
    'Omega_b':        0.04897,
    'Omega_cdm':      0.26069 - 0.04897,   # Omega_m - Omega_b
    'Omega_m':        0.26069,
    'n_s':            0.9665,
    'ln10^{10}A_s':   3.096,               # ln(10^10 * 2.209e-9)
    'w0':             -1.0,
    'wa':              0.0,
}

print('Setting up LinearPowerSpectrum (CAMB)...')
pk_lin = LinearPowerSpectrum()
print('Setting up HaloMassFunction (Tinker 2008)...')
hmf = HaloMassFunction(pk_lin.pk_linear)

# HOD model (Zheng+2007)
hod_model = HODModel(hmf, hmf.bias)

# HOD parameters: SDSS Main Sample-like for log10 M* > 10.0 (Zheng+2007)
hod_params = {
    'log10mmin':  11.35,   # log10(M_min / [Msun/h])
    'sigma_logm':  0.25,   # scatter in central HOD
    'log10m0':    11.20,   # satellite suppression mass
    'log10m1':    12.40,   # satellite mass scale
    'alpha':       1.0,    # satellite power-law slope
}

# Effective redshift
z_eff = np.sum(z_centres * nz_norm)
print(f'\nEffective redshift: z_eff = {z_eff:.3f}')

# Get effective bias from HOD model
n_gal_hod, b_eff, m_eff = hod_model._integrate(float(z_eff), theta_cosmo, hod_params)
print(f'HOD predictions at z = {z_eff:.3f}:')
print(f'  n_gal     = {n_gal_hod:.3e} Mpc^-3 h^3')
print(f'  b_eff     = {b_eff:.3f}')
print(f'  m_eff     = {m_eff:.2e} Msun/h')

In [ ]:
# Limber approximation: C_ell = int H(z)/c * [n(z)/nbar]^2 * b^2 * P_lin(l/chi) / chi^2 dz

# We compute P_lin on a k grid and interpolate
k_grid = np.logspace(-3, 1.5, 300)  # h/Mpc
pk_z   = pk_lin.pk_linear(k_grid, float(z_eff), theta_cosmo)  # (Mpc/h)^3
log_k  = np.log(k_grid)
log_pk = np.log(pk_z)

# Geometric factors on the n(z) z-grid
h_factor = cosmo_astropy.H(z_centres).to('1/s').value * 3.0857e22  # 1/s -> km/s/Mpc -> 1/Mpc
# Actually use H(z) in km/s/Mpc and c in km/s
c_light  = 299792.458  # km/s
H_z      = cosmo_astropy.H(z_centres).value  # km/s/Mpc
chi_z    = cosmo_astropy.comoving_distance(z_centres).value  # Mpc

# n(z) selection function
W_z = nz_norm / dz  # per unit redshift, normalised so int W_z dz = 1

# Compute C_ell for each ell bin
ell_theory = ell_eff.copy()
cl_theory  = np.zeros(len(ell_theory))

for i, ell in enumerate(ell_theory):
    k_l = (ell + 0.5) / chi_z  # Mpc^{-1}, Limber: k = (ell+0.5)/chi

    # Convert k from Mpc^{-1} to h/Mpc
    h = theta_cosmo['h']
    k_l_h = k_l * h  # h/Mpc

    # Interpolate P_lin at each k_l(z), clamp to grid boundaries
    pk_interp = np.exp(np.interp(np.log(k_l_h), log_k, log_pk,
                                  left=log_pk[0], right=log_pk[-1]))

    # Convert P_lin from (Mpc/h)^3 to Mpc^3
    pk_mpc3 = pk_interp / h**3

    # Limber integrand: H(z)/c * W^2 * b^2 * P_lin / chi^2
    # using b_eff from HOD (constant across z for this narrow range)
    integrand = (H_z / c_light) * W_z**2 * b_eff**2 * pk_mpc3 / chi_z**2

    # Integrate over z (trapezoidal rule)
    cl_theory[i] = np.trapezoid(integrand, z_centres)

print(f'Limber C_ell computed for {len(ell_theory)} ell bins.')
print(f'\nTheory range: C_ell in [{cl_theory[cl_theory>0].min():.2e}, {cl_theory.max():.2e}]')

---

## 7. Final comparison: data vs. HOD theory

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 10),
                          gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.05},
                          sharex=True)

ax = axes[0]

# ---- Data ----
norm = ell_eff * (ell_eff + 1) / (2 * np.pi)
good = cl_data > 0

ax.errorbar(ell_eff[good], (norm * cl_data)[good], yerr=(norm * cl_err)[good],
            fmt='o', color='steelblue', ms=7, capsize=4, zorder=3,
            label='Measured $C_\\ell$ (BGS LS10, $\\log_{10}M_* > 10.0$)')

# ---- Theory (Limber + HOD bias) ----
norm_th = ell_theory * (ell_theory + 1) / (2 * np.pi)
good_th = cl_theory > 0

ax.plot(ell_theory[good_th], (norm_th * cl_theory)[good_th], '-', color='tomato', lw=2.5,
        label=f'HOD theory (Limber, $b_{{\\rm eff}} = {b_eff:.2f}$)')

# Show 20% band around theory (rough systematic estimate)
ax.fill_between(ell_theory[good_th],
                0.8 * (norm_th * cl_theory)[good_th],
                1.2 * (norm_th * cl_theory)[good_th],
                alpha=0.2, color='tomato', label='±20% theory band')

# Shot noise
ax.plot(ell_eff, norm * nl_data, 'k:', lw=1.2, label='Shot noise $N_\\ell$')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_ylabel(r'$\ell(\ell+1)C_\ell/(2\pi)$', fontsize=14)
ax.set_title('Angular Power Spectrum $C_\\ell$ of BGS Galaxies\n'
             r'$\log_{10}M_* > 10.0\,M_\odot$,  $0.05 < z < 0.18$  (Comparat et al. 2025)',
             fontsize=12)
ax.legend(fontsize=10)
ax.set_xlim(8, 1200)

# Add scale labels on top axis
ax_top = ax.twiny()
ax_top.set_xscale('log')
ax_top.set_xlim(ax.get_xlim())
ell_ticks = np.array([10, 30, 100, 300, 1000])
ax_top.set_xticks(ell_ticks)
ax_top.set_xticklabels([f'{180/l:.1f}°' for l in ell_ticks])
ax_top.set_xlabel('Angular scale $\\theta \\approx 180°/\\ell$', fontsize=11)

# ---- Residual panel ----
ax2 = axes[1]
both = good & good_th
ratio = cl_data[both] / cl_theory[both]
ratio_err = cl_err[both] / cl_theory[both]

ax2.axhline(1.0, color='tomato', lw=2)
ax2.axhline(1.2, color='tomato', ls='--', lw=1)
ax2.axhline(0.8, color='tomato', ls='--', lw=1)
ax2.errorbar(ell_eff[both], ratio, yerr=ratio_err,
             fmt='o', color='steelblue', ms=6, capsize=3, zorder=3)
ax2.set_ylabel(r'Data / Theory')
ax2.set_xlabel(r'Multipole $\ell$', fontsize=14)
ax2.set_ylim(0.0, 3.0)
ax2.set_xlim(8, 1200)

plt.savefig('aps_cl_comparison.pdf', bbox_inches='tight')
plt.show()
print('Figure saved to aps_cl_comparison.pdf')

---

## 8. Connection to the angular two-point correlation function w(θ)

The angular correlation function $w(\theta)$ and the angular power spectrum $C_\ell$ are a Legendre transform pair:

$$w(\theta) = \sum_{\ell=0}^\infty \frac{2\ell+1}{4\pi}\,C_\ell\,P_\ell(\cos\theta)$$

$$C_\ell = 2\pi \int_0^\pi w(\theta)\,P_\ell(\cos\theta)\,\sin\theta\,d\theta$$

In the flat-sky approximation (valid for $\ell \gg 1$, $\theta \ll 1$):

$$w(\theta) \approx \frac{1}{2\pi}\int_0^\infty \ell\,C_\ell\,J_0(\ell\theta)\,d\ell$$

Here we use `hod_mod.galaxies.clustering.HODClusteringPrediction.w_theta()` to predict $w(\theta)$ and `sum_stat.twopcf.angular.w_theta()` to measure it (both use the same Landy-Szalay estimator).

> ⚠️ **Note:** Measuring $w(\theta)$ with treecorr on 2.7M galaxies can take several minutes. This cell uses a subsample for speed.

In [ ]:
# Theory w(theta) from HOD model
from hod_mod.galaxies.clustering import HODClusteringPrediction

print('Setting up HODClusteringPrediction...')
predictor = HODClusteringPrediction(pk_lin, hod_model, k_min=1e-3, k_max=15.0, n_k=256)

# Angular separation bins: 0.1 to 10 degrees (arcmin)
theta_bins_arcmin = np.logspace(np.log10(1), np.log10(600), 22)  # 1 to 600 arcmin
theta_bins_deg = theta_bins_arcmin / 60.0
theta_centres_deg = np.sqrt(theta_bins_deg[:-1] * theta_bins_deg[1:])

print(f'Computing theoretical w(theta) at z_eff = {z_eff:.3f}...')

# w_theta from HOD (Limber projection, Coupon+2015 App. B)
# n_z is a tuple (z_centres, normalised n(z))
w_theory = predictor.w_theta(
    theta_deg   = theta_centres_deg,
    z           = float(z_eff),
    theta_cosmo = theta_cosmo,
    hod_params  = hod_params,
    n_z         = (z_centres, nz_norm / dz),  # (z_grid, n(z) per unit z)
    pi_max_h    = 300.0,
    n_z_steps   = 32,
)
print('Done!')
print(f'w(theta) range: [{w_theory.min():.3e}, {w_theory.max():.3e}]')

In [ ]:
from sum_stat.twopcf.angular import w_theta as measure_w_theta

# Use a subsample of 200,000 galaxies for speed
N_WTHETA = 200_000
rng = np.random.default_rng(7)
idx_sub = rng.choice(len(ra_g), size=N_WTHETA, replace=False)

cat_sub = GalaxyCatalogue(
    ra       = ra_g[idx_sub],
    dec      = dec_g[idx_sub],
    redshift = z_g[idx_sub],
    weight   = np.ones(N_WTHETA),
)
rand_sub = GalaxyCatalogue(
    ra       = rand_ra_g[:N_WTHETA * 5],
    dec      = rand_dec_g[:N_WTHETA * 5],
    redshift = np.zeros(N_WTHETA * 5),  # not used for angular
)

print(f'Measuring w(theta) on {N_WTHETA:,} galaxy subsample...')
theta_data, w_data, varw_data = measure_w_theta(
    gal       = cat_sub,
    rand      = rand_sub,
    theta_bins= theta_bins_arcmin,
    estimator = 'landy-szalay',
    sep_units = 'arcmin',
    n_threads = 4,
)
w_err_data = np.sqrt(varw_data)
print('Done!')

# Plot w(theta)
fig, ax = plt.subplots(figsize=(9, 6))

ax.errorbar(theta_data, theta_data * w_data, yerr=theta_data * w_err_data,
            fmt='o', color='steelblue', ms=6, capsize=4,
            label=f'Measured $w(\\theta)$ (Landy-Szalay, {N_WTHETA//1000}k galaxies)')
ax.plot(theta_centres_deg * 60, theta_centres_deg * 60 * np.array(w_theory),
        '-', color='tomato', lw=2,
        label=f'HOD theory (Limber, $b_{{\\rm eff}} = {b_eff:.2f}$)')

ax.set_xscale('log')
ax.set_xlabel(r'Angular separation $\theta$ [arcmin]', fontsize=14)
ax.set_ylabel(r'$\theta \cdot w(\theta)$', fontsize=14)
ax.set_title(r'Angular Correlation Function $w(\theta)$ — BGS LS10')
ax.axhline(0, color='k', lw=0.8)
ax.legend()

# Secondary x-axis in degrees
ax2x = ax.twiny()
ax2x.set_xscale('log')
ax2x.set_xlim(np.array(ax.get_xlim()) / 60)
ax2x.set_xlabel(r'$\theta$ [degrees]', fontsize=11)

plt.savefig('aps_wtheta_comparison.pdf', bbox_inches='tight')
plt.show()
print('Figure saved to aps_wtheta_comparison.pdf')

---

## 9. Discussion and summary

### 9.1 Interpreting the power spectrum

The angular power spectrum $C_\ell$ contains a wealth of information:

- **Large scales ($\ell < 100$, $\theta > 2°$):** Dominated by **two-halo clustering** — the large-scale structure of the cosmic web. The signal is proportional to $b_{\rm eff}^2\,P_{\rm lin}(k)$ and follows the shape of the matter power spectrum, including the BAO peak at $\ell \sim 300$ for $z \sim 0.1$.

- **Small scales ($\ell > 300$, $\theta < 0.5°$):** Dominated by **one-halo clustering** — galaxies within the same dark matter halo. At these scales, we see the effect of satellite galaxies orbiting within their host halos. The 2-halo theory (used here) underestimates the signal here.

- **Very small scales ($\ell > 800$):** The signal approaches the **shot noise** floor $N_\ell = \Omega_{\rm pix}/N_{\rm gal}$.

### 9.2 Systematic effects

The main systematic effects in the angular power spectrum measurement are:

1. **Integral constraint:** On a finite survey of area $\Omega$, the measured mean $\bar{n}$ is itself an estimate of the true mean, which removes the $\ell=0$ mode and suppresses power at $\ell \lesssim \pi/\sqrt{\Omega}$.

2. **Mask effects:** The survey boundary introduces mode coupling (off-diagonal $C_\ell$ correlations) not captured by the simple $f_{\rm sky}$ correction. The full MASTER correction (Hivon et al. 2002) accounts for this.

3. **Photometric redshift scatter:** Photo-z errors with $\sigma_z \sim 0.01(1+z)$ wash out the radial (3D) power spectrum, projecting more signal into the angular $C_\ell$.

4. **Stellar contamination:** Stars misidentified as galaxies add a spurious signal correlated with the Galactic plane.

5. **Dust extinction:** Regions of high $E(B-V)$ have lower galaxy completeness, introducing a spurious angular pattern.

### 9.3 Agreement with theory

At large scales ($\ell < 100$), the data should agree with the 2-halo HOD theory to $\sim 10-20\%$ if the HOD parameters are correct. Significant deviations may indicate:
- Wrong HOD parameters (adjust $\log_{10}M_{\rm min}$, $b_{\rm eff}$)
- Systematics in the data (dust, stellar contamination)
- Missing physics in the model (scale-dependent bias, baryonic effects)

### 9.4 Further reading

- **Comparat et al. (2025):** [A&A 697, A173](https://doi.org/10.1051/0004-6361/202554208) — this dataset and cross-correlations with X-rays
- **More et al. (2015):** ApJ 806, 2 — HOD clustering predictions with 2-halo approximation
- **Zheng et al. (2007):** ApJ 667, 760 — HOD parameterisation
- **Johnston (2011):** [arXiv:1106.2039](https://arxiv.org/abs/1106.2039) — review of estimators and methods
- **Hivon et al. (2002):** ApJ 567, 2 — MASTER algorithm for pseudo-$C_\ell$

### 9.5 Exercises

1. **Change the HOD parameters:** Increase $\log_{10}M_{\rm min}$ to 11.8 and observe the effect on $b_{\rm eff}$ and the predicted $C_\ell$.

2. **Change the stellar mass threshold:** Redo the measurement for a more massive sample ($\log_{10}M_* > 11.0$) — how does the amplitude of $C_\ell$ change? What does this tell you about the galaxy bias?

3. **Pixelisation resolution:** Try `NSIDE=256` and `NSIDE=1024`. How does the shot noise and the maximum $\ell$ change?

4. **Mask apodisation:** Apply a Gaussian apodisation to the mask edges and compare with the hard mask. Does it change the power spectrum at large scales?

In [ ]:
print('=' * 70)
print('SUMMARY — Angular Power Spectrum of BGS Galaxies')
print(f'Sample: log10 M* > {MSTAR_LIMIT}, {Z_MIN} < z < {Z_MAX}')
print(f'N_gal     = {len(ra_g):,}')
print(f'Area      = {area_deg2:,.0f} deg²')
print(f'f_sky     = {f_sky:.4f}')
print(f'N_side    = {NSIDE}  =>  ell_max ~ {2*NSIDE}')
print(f'Shot noise N_ell = {nl_data[0]:.2e} sr')
print(f'HOD bias b_eff   = {b_eff:.3f} (Zheng+2007 params, z_eff={z_eff:.2f})')
print('=' * 70)
print(f'\n{"ell_eff":>8s}  {"C_ell [sr]":>14s}  {"N_ell [sr]":>14s}  {"S/N":>6s}')
print('-' * 60)
for le, cl, nl, se in zip(ell_eff, cl_data, nl_data, cl_err):
    snr = cl/se if se > 0 and cl > 0 else 0
    print(f'{le:8.1f}  {cl:14.3e}  {nl:14.3e}  {snr:6.1f}')
print('=' * 70)